In [1]:
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

## 1、加载数据集

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "tyqiangz/multilingual-sentiments",
    name="chinese",
    cache_dir="../data",
    trust_remote_code=True 
)


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'source', 'label'],
        num_rows: 120000
    })
    validation: Dataset({
        features: ['text', 'source', 'label'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['text', 'source', 'label'],
        num_rows: 3000
    })
})

In [ ]:
dataset['train'][0]

{'label': tensor(2),
 'input_ids': tensor([ 101, 3315,  782, 6572, 1384, 6158, 4668, 8024, 6598, 7032, 6158, 3736,
         6205, 8020, 3342, 2456, 8021, 2918, 4500, 8024, 6435,  762, 7716, 6849,
         2226, 2571, 3389, 2141, 8024, 2199, 3315,  782, 4638, 8185, 1039, 6598,
         7032, 6842, 1726,  511, 3315,  782, 2347,  754, 8109, 2399, 8111, 3299,
         8114, 3189, 2990,  769, 6842, 6573, 4509, 6435, 8024,  711,  862, 1168,
         8271, 2399,  749, 6820, 3221, 3766, 6237, 1104, 8043,  762, 7716, 6849,
         3221,  784,  720, 2658, 1105, 8043, 6435, 5314, 3315,  782,  671,  702,
         1394, 4415, 6237, 7025,  511,  102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor(

In [5]:
(dataset.num_rows,
dataset["train"].info,
dataset["train"].info.features)


({'train': 120000, 'validation': 3000, 'test': 3000},
 DatasetInfo(description='chinese dataset.', citation='chinese citation', homepage='', license='', features={'text': Value(dtype='string', id=None), 'source': Value(dtype='string', id=None), 'label': ClassLabel(names=['positive', 'neutral', 'negative'], id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name='parquet', dataset_name='multilingual-sentiments', config_name='chinese', version=1.0.0, splits={'train': SplitInfo(name='train', num_bytes=22489086, num_examples=120000, shard_lengths=None, dataset_name='multilingual-sentiments'), 'validation': SplitInfo(name='validation', num_bytes=556160, num_examples=3000, shard_lengths=None, dataset_name='multilingual-sentiments'), 'test': SplitInfo(name='test', num_bytes=563897, num_examples=3000, shard_lengths=None, dataset_name='multilingual-sentiments')}, download_checksums={'hf://datasets/tyqiangz/multilingual-sentiments@be62509a25678d289e65505e4ebba4519

## 2、数据预处理

In [6]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese', cache_dir='./models')
tokenizer

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BertTokenizerFast(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [7]:
def tokenization(datasets):
    return tokenizer(datasets["text"], max_length=128, truncation=True)

In [8]:

dataset = dataset.map(tokenization, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    validation: Dataset({
        features: ['text', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['text', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
})

In [9]:
print(dataset["train"][0])

{'label': tensor(2), 'input_ids': tensor([ 101, 3315,  782, 6572, 1384, 6158, 4668, 8024, 6598, 7032, 6158, 3736,
        6205, 8020, 3342, 2456, 8021, 2918, 4500, 8024, 6435,  762, 7716, 6849,
        2226, 2571, 3389, 2141, 8024, 2199, 3315,  782, 4638, 8185, 1039, 6598,
        7032, 6842, 1726,  511, 3315,  782, 2347,  754, 8109, 2399, 8111, 3299,
        8114, 3189, 2990,  769, 6842, 6573, 4509, 6435, 8024,  711,  862, 1168,
        8271, 2399,  749, 6820, 3221, 3766, 6237, 1104, 8043,  762, 7716, 6849,
        3221,  784,  720, 2658, 1105, 8043, 6435, 5314, 3315,  782,  671,  702,
        1394, 4415, 6237, 7025,  511,  102]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 

## 3、创建模型

In [10]:
from transformers import AutoModelForSequenceClassification 
model = AutoModelForSequenceClassification.from_pretrained('bert-base-chinese', cache_dir='./models', num_labels=3)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## 4、创建评估函数

In [11]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metirc = evaluate.load("f1")

In [12]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels, average="micro")
    acc.update(f1)
    return acc

## 5、创建训练器

In [13]:
train_args = TrainingArguments(
    output_dir="./Sentiment_model",      # 输出文件夹
    per_device_train_batch_size=128,  # 训练时的batch_size
    per_device_eval_batch_size=256,   # 验证时的batch_size
    logging_steps=50,                # log 打印的频率
    evaluation_strategy="epoch",     # 评估策略
    save_strategy="epoch",           # 保存策略
    save_total_limit=3,              # 最大保存数
    num_train_epochs=3,              # 训练轮数, 默认为3
    learning_rate=2e-5,              # 学习率
    weight_decay=0.01,               # weight_decay
    metric_for_best_model="f1",      # 设定评估指标
    report_to=['tensorboard'],       # tensorboard展示结果
    load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

## 6、训练模型

In [14]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model,
                  args=train_args,
                  train_dataset=dataset["train"],
                  eval_dataset=dataset["validation"],
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [15]:
trainer.train()

  0%|          | 0/2814 [00:00<?, ?it/s]

{'loss': 0.7182, 'grad_norm': 6.476371765136719, 'learning_rate': 1.9644633972992183e-05, 'epoch': 0.05}
{'loss': 0.5639, 'grad_norm': 3.189556837081909, 'learning_rate': 1.9289267945984364e-05, 'epoch': 0.11}
{'loss': 0.5502, 'grad_norm': 3.8066980838775635, 'learning_rate': 1.8933901918976546e-05, 'epoch': 0.16}
{'loss': 0.537, 'grad_norm': 5.166387557983398, 'learning_rate': 1.857853589196873e-05, 'epoch': 0.21}
{'loss': 0.5271, 'grad_norm': 2.9593214988708496, 'learning_rate': 1.822316986496091e-05, 'epoch': 0.27}
{'loss': 0.5154, 'grad_norm': 2.866473913192749, 'learning_rate': 1.7867803837953093e-05, 'epoch': 0.32}
{'loss': 0.5238, 'grad_norm': 3.989497661590576, 'learning_rate': 1.7512437810945274e-05, 'epoch': 0.37}
{'loss': 0.5136, 'grad_norm': 4.483031272888184, 'learning_rate': 1.715707178393746e-05, 'epoch': 0.43}
{'loss': 0.5234, 'grad_norm': 2.446608304977417, 'learning_rate': 1.6801705756929637e-05, 'epoch': 0.48}
{'loss': 0.5234, 'grad_norm': 3.247462749481201, 'learnin

  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 0.5074107050895691, 'eval_accuracy': 0.7866666666666666, 'eval_f1': 0.7866666666666666, 'eval_runtime': 8.3505, 'eval_samples_per_second': 359.262, 'eval_steps_per_second': 1.437, 'epoch': 1.0}
{'loss': 0.4786, 'grad_norm': 2.6440317630767822, 'learning_rate': 1.3248045486851457e-05, 'epoch': 1.01}
{'loss': 0.4391, 'grad_norm': 3.044182062149048, 'learning_rate': 1.289267945984364e-05, 'epoch': 1.07}
{'loss': 0.4529, 'grad_norm': 3.117818593978882, 'learning_rate': 1.2537313432835823e-05, 'epoch': 1.12}
{'loss': 0.4366, 'grad_norm': 2.635568857192993, 'learning_rate': 1.2181947405828003e-05, 'epoch': 1.17}
{'loss': 0.4416, 'grad_norm': 3.346970558166504, 'learning_rate': 1.1826581378820186e-05, 'epoch': 1.23}
{'loss': 0.4546, 'grad_norm': 3.4391605854034424, 'learning_rate': 1.1471215351812369e-05, 'epoch': 1.28}
{'loss': 0.4362, 'grad_norm': 4.591310977935791, 'learning_rate': 1.111584932480455e-05, 'epoch': 1.33}
{'loss': 0.4353, 'grad_norm': 3.0463433265686035, 'learni

  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 0.506606936454773, 'eval_accuracy': 0.7916666666666666, 'eval_f1': 0.7916666666666666, 'eval_runtime': 7.9006, 'eval_samples_per_second': 379.718, 'eval_steps_per_second': 1.519, 'epoch': 2.0}
{'loss': 0.4203, 'grad_norm': 3.4555182456970215, 'learning_rate': 6.496090973702914e-06, 'epoch': 2.03}
{'loss': 0.3984, 'grad_norm': 3.0715911388397217, 'learning_rate': 6.140724946695097e-06, 'epoch': 2.08}
{'loss': 0.3996, 'grad_norm': 3.210052490234375, 'learning_rate': 5.785358919687279e-06, 'epoch': 2.13}
{'loss': 0.3872, 'grad_norm': 3.3084847927093506, 'learning_rate': 5.42999289267946e-06, 'epoch': 2.19}
{'loss': 0.3891, 'grad_norm': 3.1905441284179688, 'learning_rate': 5.074626865671642e-06, 'epoch': 2.24}
{'loss': 0.3875, 'grad_norm': 3.1184520721435547, 'learning_rate': 4.719260838663824e-06, 'epoch': 2.29}
{'loss': 0.3934, 'grad_norm': 3.983818531036377, 'learning_rate': 4.363894811656006e-06, 'epoch': 2.35}
{'loss': 0.3835, 'grad_norm': 3.0047104358673096, 'learning_r

  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 0.5242984294891357, 'eval_accuracy': 0.7913333333333333, 'eval_f1': 0.7913333333333333, 'eval_runtime': 7.9177, 'eval_samples_per_second': 378.896, 'eval_steps_per_second': 1.516, 'epoch': 3.0}
{'train_runtime': 2733.0588, 'train_samples_per_second': 131.721, 'train_steps_per_second': 1.03, 'train_loss': 0.451103926382763, 'epoch': 3.0}


TrainOutput(global_step=2814, training_loss=0.451103926382763, metrics={'train_runtime': 2733.0588, 'train_samples_per_second': 131.721, 'train_steps_per_second': 1.03, 'total_flos': 2.367895780678579e+16, 'train_loss': 0.451103926382763, 'epoch': 3.0})

## 7、评估

In [16]:
trainer.evaluate(dataset["test"])

  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 0.47254595160484314,
 'eval_accuracy': 0.815,
 'eval_f1': 0.815,
 'eval_runtime': 7.8941,
 'eval_samples_per_second': 380.029,
 'eval_steps_per_second': 1.52,
 'epoch': 3.0}

## 8、预测

In [17]:
from transformers import pipeline

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [18]:
id2label_dic = {
    0:"积极",
    1:"中性",
    2:"消极"
                }
model.config.id2label = id2label_dic


In [19]:
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [26]:
sen = "这个电影不错，我很喜欢"
pipe(sen)

[{'label': '积极', 'score': 0.9599310755729675}]

In [27]:
sen = "这味道太差了"
pipe(sen)


[{'label': '消极', 'score': 0.9799193143844604}]

In [28]:
sen = "1+2=3"
pipe(sen)


[{'label': '中性', 'score': 0.472244530916214}]